# AI Subscription Economy: From Messy Data to a Reproducible Preprocessing Pipeline

**Session:** S02 — Data Wrangling and Exploratory Data Analysis  
**Estimated time:** 90–120 minutes | **Format:** Guided tutorial

## Scenario
A fictional analytics team wants to understand how people use AI subscriptions and prepare a reliable feature table for a later machine-learning task. The dataset is synthetic and intentionally contains missing values, inconsistent categories, duplicated records, malformed numeric values and unusual observations.

**This tutorial teaches data preparation, not predictive model training.** S03 introduces predictive modelling.

## Learning outcomes
1. Inspect a tabular dataset and explain its schema and quality issues.
2. Apply and justify transparent cleaning rules.
3. Use descriptive statistics and plots for exploratory data analysis.
4. Create meaningful features while avoiding target leakage.
5. Build a scikit-learn preprocessing pipeline fitted on training data only.

Synthetic data do not support claims about real AI-tool users, market shares, fairness or customer behaviour.

## 0. Environment and imports

Run the notebook from top to bottom. No API key, external dataset, GPU or paid service is required.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

## 1. Generate the fictional dataset

We create 1,200 synthetic user snapshots and deliberately inject data-quality problems. One row represents one fictional user snapshot. future_spend is a possible future outcome, not a feature available for a decision made today.

In [ ]:
rng = np.random.default_rng(SEED)
n = 1200
tools = np.array(["ChatGPT", "Claude", "Gemini", "Copilot"])
countries = np.array(["Malta", "Italy", "Spain", "Germany", "France", "UK"])
plans = np.array(["Free", "Pro", "Team", "Enterprise"])
uses = np.array(["coding", "writing", "research", "productivity"])

raw = pd.DataFrame({
    "user_id": [f"U{i:05d}" for i in range(n)],
    "age": rng.integers(18, 71, n).astype(object),
    "country": rng.choice(countries, n).astype(object),
    "subscription_type": rng.choice(plans, n).astype(object),
    "ai_tool": rng.choice(tools, n).astype(object),
    "monthly_spend": np.round(rng.gamma(2.0, 12.0, n), 2).astype(object),
    "weekly_sessions": rng.poisson(8, n).astype(object),
    "avg_session_minutes": np.round(rng.gamma(2.5, 18.0, n), 1).astype(object),
    "primary_use": rng.choice(uses, n).astype(object),
    "satisfaction_score": rng.integers(1, 6, n).astype(object),
    "subscription_months": rng.integers(0, 49, n).astype(object),
    "future_spend": np.round(rng.gamma(2.1, 13.0, n), 2)
})

for col, count in [("age", 35), ("country", 42), ("monthly_spend", 48),
                   ("primary_use", 30), ("satisfaction_score", 25)]:
    raw.loc[rng.choice(raw.index, count, replace=False), col] = np.nan

for idx in rng.choice(raw.index, 55, replace=False):
    raw.loc[idx, "ai_tool"] = rng.choice(["chatgpt", "CHAT GPT", "claude ", "GEMINI"])
for idx in rng.choice(raw.index, 22, replace=False):
    raw.loc[idx, "monthly_spend"] = rng.choice(["unknown", "€19.99", "N/A"])
for idx in rng.choice(raw.index, 8, replace=False):
    raw.loc[idx, "age"] = rng.choice([-4, 0, 150, "not recorded"])
for idx in rng.choice(raw.index, 8, replace=False):
    raw.loc[idx, "monthly_spend"] = -abs(float(rng.normal(20, 5)))
for idx in rng.choice(raw.index, 5, replace=False):
    raw.loc[idx, "weekly_sessions"] = 250

raw = pd.concat([raw, raw.sample(12, random_state=SEED)], ignore_index=True)
raw.to_csv("ai_subscription_raw.csv", index=False)
print("Dataset shape:", raw.shape)
display(raw.head(10))

### Checkpoint 1
- What does one row represent?
- Which columns are identifiers, candidate features and possible outcomes?
- Why is future_spend not an input feature for a same-day prediction?
- What evidence would be needed before generalising to real users?

## 2. Inspect structure and types

Inspect dimensions, sample rows, column names and inferred types before cleaning.

In [ ]:
print("Rows, columns:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
raw.info()

**Think:** Which columns have unexpected data types? Why might numeric-looking values be stored as objects?

## 3. Audit missing values

A null count tells us where values are absent, not why. Missingness may depend on observed or unobserved processes. Do not infer MCAR, MAR or MNAR from a summary table alone.

In [ ]:
missing = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(1)
}).sort_values("missing_percent", ascending=False)
display(missing)

**Exercise:** Choose two columns with missing values. Suggest a plausible reason for each and identify evidence needed to test your explanation.

## 4. Detect duplicate records

An identical row may be a repeated export. A repeated user_id alone is not necessarily a duplicate: a real system may contain multiple snapshots per user. Here, 12 exact duplicate rows were deliberately appended.

In [ ]:
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Repeated user IDs:", int(raw["user_id"].duplicated().sum()))

clean = raw.drop_duplicates().copy()
print("Shape after exact deduplication:", clean.shape)

**Checkpoint:** Why is dropping every repeated user_id not automatically safe? Define the unit of observation before choosing a deduplication key.

## 5. Standardise categorical values

Inspect observed categories first. Apply explicit mappings rather than blindly lowercasing all strings.

In [ ]:
for col in ["ai_tool", "country", "subscription_type", "primary_use"]:
    print("\n", col)
    print(clean[col].value_counts(dropna=False).head(20))

In [ ]:
tool_map = {
    "chatgpt": "ChatGPT", "chat gpt": "ChatGPT",
    "claude": "Claude", "gemini": "Gemini", "copilot": "Copilot"
}
clean["ai_tool"] = (
    clean["ai_tool"].astype("string").str.strip().str.lower()
    .map(tool_map).fillna(clean["ai_tool"])
)
clean["country"] = clean["country"].astype("string").str.strip()
clean["subscription_type"] = clean["subscription_type"].astype("string").str.strip().str.title()
clean["primary_use"] = clean["primary_use"].astype("string").str.strip().str.lower()
display(clean["ai_tool"].value_counts(dropna=False))

**Exercise:** How should unknown categories be handled: mapped, preserved as a category, or marked missing? Defend your choice.

## 6. Convert numeric values and validate domain rules

Parse currency symbols explicitly, coerce malformed values to missing, and then apply documented domain checks. A flagged value is not necessarily safe to delete.

In [ ]:
def parse_money(series):
    text = series.astype("string").str.replace("€", "", regex=False).str.strip()
    return pd.to_numeric(text, errors="coerce")

numeric_cols = [
    "age", "monthly_spend", "weekly_sessions", "avg_session_minutes",
    "satisfaction_score", "subscription_months", "future_spend"
]
clean["monthly_spend"] = parse_money(clean["monthly_spend"])
for col in numeric_cols:
    if col != "monthly_spend":
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

display(clean[numeric_cols].describe().T)

In [ ]:
# Flag invalid values. This tutorial's policy is to mark them missing, not delete rows.
invalid_age = ~clean["age"].between(13, 100)
invalid_spend = clean["monthly_spend"] < 0
invalid_sessions = clean["weekly_sessions"] < 0
invalid_duration = clean["avg_session_minutes"] <= 0
invalid_satisfaction = ~clean["satisfaction_score"].between(1, 5)

for label, mask in [
    ("Invalid age", invalid_age), ("Negative spend", invalid_spend),
    ("Negative sessions", invalid_sessions), ("Non-positive duration", invalid_duration),
    ("Satisfaction outside 1–5", invalid_satisfaction)
]:
    print(label + ":", int(mask.sum()))

clean.loc[invalid_age, "age"] = np.nan
clean.loc[invalid_spend, "monthly_spend"] = np.nan
clean.loc[invalid_sessions, "weekly_sessions"] = np.nan
clean.loc[invalid_duration, "avg_session_minutes"] = np.nan
clean.loc[invalid_satisfaction, "satisfaction_score"] = np.nan

**Discussion:** A weekly count of 250 is unusual but may be possible depending on the session definition. Why investigate rather than automatically delete it? In a real billing system, could negative spend represent a refund?

## 7. Exploratory data analysis

EDA describes distributions and relationships and generates questions. It does not establish causality. Always interpret axes, units, selection and unusual values.

In [ ]:
display(clean.groupby("subscription_type", dropna=False)
        .agg(records=("user_id", "count"),
             median_spend=("monthly_spend", "median"),
             mean_sessions=("weekly_sessions", "mean"))
        .sort_values("records", ascending=False))
display(clean["ai_tool"].value_counts(dropna=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
clean["monthly_spend"].dropna().plot(kind="hist", bins=30, ax=ax)
ax.set(title="Monthly spend distribution (synthetic records)",
       xlabel="Monthly spend (EUR)", ylabel="Record count")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
clean["ai_tool"].value_counts().plot(kind="bar", ax=ax)
ax.set(title="Records by AI tool (synthetic data)",
       xlabel="AI tool", ylabel="Record count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
sample = clean.dropna(subset=["weekly_sessions", "monthly_spend"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sample["weekly_sessions"], sample["monthly_spend"], alpha=.45)
ax.set(title="Weekly sessions vs monthly spend",
       xlabel="Weekly sessions (count)", ylabel="Monthly spend (EUR)")
plt.tight_layout()
plt.show()

### EDA questions
1. Which plan has the most records? Does that prove it is most popular in the real market?
2. Describe the spend distribution. Which is more robust to extreme values: mean or median?
3. Is there an association between sessions and spend? What confounders may matter?
4. Why does correlation not establish causation?

## 8. Feature engineering

Create spend_per_session. It is undefined when weekly_sessions is zero or missing, so division by zero is explicitly avoided.

In [ ]:
clean["spend_per_session"] = (
    clean["monthly_spend"] / clean["weekly_sessions"].replace(0, np.nan)
)
display(clean[["monthly_spend", "weekly_sessions", "spend_per_session"]].head(10))

**Target leakage:** future_spend is a possible outcome, not an input feature. It is deliberately excluded from X because it would not be available at prediction time.

## 9. Split before fitting transformations

Reserve a holdout partition before fitting any imputer, scaler or encoder. A random split is used only for this introductory demonstration. Repeated user snapshots may require group-aware or time-based splitting.

In [ ]:
target = "future_spend"
excluded = ["user_id", target]
feature_cols = [col for col in clean.columns if col not in excluded]

X = clean[feature_cols].copy()
# Convert pandas nullable string columns to object with np.nan so SimpleImputer can process missing values.
for col in X.select_dtypes(include="string").columns:
    X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
y = clean[target].copy()  # Kept separate; no predictive model is trained in S02.

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print("Training rows:", len(X_train))
print("Holdout rows:", len(X_holdout))
print("Feature columns:", X_train.columns.tolist())

## 10. Build a train-only preprocessing pipeline

Numeric features: median imputation then standardisation. Categorical features: most-frequent imputation then one-hot encoding. Unknown categories in holdout are ignored safely.

The pipeline learns medians, means, standard deviations and category vocabulary only when fit is called on X_train.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_holdout_prepared = preprocessor.transform(X_holdout)
print("Training matrix:", X_train_prepared.shape)
print("Holdout matrix:", X_holdout_prepared.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Transformed feature count:", len(feature_names))
print(feature_names.tolist()[:40])

### Why this order matters

Incorrect: fit imputation, scaling or encoding on all rows, then split. The holdout has influenced learned preprocessing parameters.

Correct: split first, fit_transform on training features, transform holdout features. This preserves the holdout's role as unseen data for later evaluation. The target is never passed to the preprocessor.

## 11. Final checks

In [ ]:
assert X_train_prepared.shape[0] == X_train.shape[0]
assert X_holdout_prepared.shape[0] == X_holdout.shape[0]
assert "future_spend" not in X_train.columns
assert "user_id" not in X_train.columns
assert len(feature_names) == X_train_prepared.shape[1]

print("Preprocessing checks passed.")
print("Raw rows:", len(raw))
print("Rows after exact deduplication:", len(clean))
print("Original feature count:", X_train.shape[1])
print("Encoded feature count:", X_train_prepared.shape[1])

## 12. Reflection and independent challenge

Answer in your own words:
1. Which cleaning decision could change the dataset most, and why?
2. Which values did you flag rather than delete? What evidence would you request?
3. Why is an exact duplicate different from a repeated user ID?
4. Which features did you engineer, and when are they available?
5. What information did the pipeline learn from training data?
6. What changes if there are multiple monthly snapshots per user?
7. What split is appropriate for predicting next month's spend?

### Independent challenge
- Add a new AI-tool category that appears only in holdout.
- Add a zero-session record and explain the derived feature.
- Compare median imputation with dropping incomplete rows.
- Propose a group-aware or time-based split for repeated snapshots.
- Write a data dictionary with units, meaning and prediction-time availability.

**Do not train a predictive model in this exercise.** S03 introduces predictive modelling with a separately defined task and evaluation protocol.

## Key takeaway
Data preparation is not mechanical. Cleaning, feature and split decisions encode assumptions. Record those assumptions, keep holdout information out of fitted transformations, and distinguish reproducible code from valid evidence about the world.